# 🌊 LFM2-8B Terminal Agent Demonstration
이 노트북은 학습된 **LFM2-8B-A1B 터미널 에이전트** 모델을 시연하고 평가하기 위한 도구입니다. Unsloth 라이브러리를 사용하여 코랩의 무료 T4 GPU 리소스에서도 원활하게 추론이 가능하도록 최적화되었습니다.

### 🚀 주요 기능
1. **학습 모델 로드**: `gyung/LFM2-8B-Terminal-SFT-Unsloth` (또는 사용자 지정 체크포인트)
2. **인터랙티브 터미널 채팅**: 자연어 명령을 입력하면 에이전트가 분석 및 셸 명령 생성
3. **벤치마크 테스트**: 10가지 핵심 터미널 시나리오 평가

---

### 1. 필수 라이브러리 설치
실행 전 상단 메뉴에서 `런타임` -> `런타임 유형 변경` -> **T4 GPU** 가 선택되어 있는지 확인하세요.

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps bitsandbytes accelerate xformers==0.0.32.post2 peft trl triton cut_cross_entropy unsloth_zoo

### 2. 모델 로드 (학습된 결과물)

In [ ]:
import transformers
import builtins
import torch
from unsloth import FastLanguageModel

# [패치] Python 환경 호환성 문제 해결
if not hasattr(builtins, "PreTrainedConfig"):
    builtins.PreTrainedConfig = transformers.PretrainedConfig

model_name = "gyung/LFM2-8B-Terminal-SFT-Unsloth" # 학습 중인 모델 리포지토리
max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # 무료 코랩 T4 GPU를 위해 4비트 양자화 사용
    trust_remote_code = True,
)
FastLanguageModel.for_inference(model) # 2배 빠른 추론 모드 활성화

### 3. 터미널 에이전트 추론 함수 정의

In [ ]:
SYSTEM_PROMPT = """You are an AI assistant tasked with solving command-line tasks in a Linux environment. You will be given a task description and the output from previously executed commands. Your goal is to solve the task by providing batches of shell commands.

Format your response as JSON with the following structure:

{
  "analysis": "Analyze the current state based on the terminal output provided.",
  "plan": "Describe your plan for the next steps.",
  "commands": [
    {"keystrokes": "ls -la\\n", "duration": 0.1}
  ],
  "task_complete": false
}"""

def chat_with_agent(user_task):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Current terminal state:\n$ \n\nTask: {user_task}"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)

    _ = model.generate(
        input_ids = inputs,
        streamer = text_streamer,
        max_new_tokens = 1024,
        use_cache = True,
        temperature = 0.3,
        min_p = 0.15,
        repetition_penalty = 1.05,
    )

### 4. 실제 에이전트와 대화하기 (테스트)

In [ ]:
user_input = "Find all files larger than 100MB in the current directory and sort them by size."
chat_with_agent(user_input)

### 5. 핵심 벤치마크 테스트 (EVAL_PROMPTS)
학습 과정에서 사용된 핵심 10가지 시나리오를 즉시 실행합니다.

In [ ]:
test_scenarios = [
    "Find all .log files modified in the last 7 days under /var/log and count lines.",
    "Extract the 3rd column from a CSV file /data/input.csv, sort, and remove duplicates.",
    "Check which Python packages are outdated and upgrade pip to the latest version.",
    "Extract users with age greater than 30 from /data/users.json using jq."
]

for i, scenario in enumerate(test_scenarios):
    print(f"\n[Scenario {i+1}] {scenario}")
    print("-"*50)
    chat_with_agent(scenario)
    print("\n")